# Pruning: magnitude, structured & 2:4 sparsity

Delete the weights that contribute least, keep the rest. Pruning is the most
intuitively obvious compression technique and the one most often measured wrong: it is
easy to remove 90% of a network's weights and achieve **exactly zero speedup**.

The distinction that decides whether pruning pays is **what shape the hole is**. This
notebook is built around that one idea. Sibling techniques:
[Distillation](knowledge-distillation.ipynb) (train a smaller model),
[Sparsity induction](sparsity-induction.ipynb) (train toward zeros in the first place),
[Low-rank factorization](low-rank-factorization.ipynb) and
[Quantization](quantization-gptq-awq.ipynb).

## 1. What & Why

Trained networks are heavily over-parameterised. A large fraction of weights sit near
zero and contribute almost nothing to the output, so removing them costs little
accuracy. That much is uncontroversial and has been known since the late 1980s.

The catch is what "removing" means. Setting a weight to zero does not make it
disappear — a dense matrix with zeros in it is still a dense matrix, and a GPU
multiplying by it does exactly as much work as before. To get a real win you need one
of three things:

1. **Structured pruning** — remove an entire row, channel, attention head or layer, so
   the weight matrix genuinely gets smaller and the matmul genuinely gets cheaper.
2. **Hardware-supported semi-structured sparsity** — the 2:4 pattern (two zeros in
   every group of four), which NVIDIA Ampere and later can execute at roughly 2× dense
   throughput via sparse tensor cores.
3. **A sparse kernel and enough sparsity to beat its overhead** — usually >95%, which
   is far past where most networks hold their accuracy.

**Reach for pruning when** you have a trained model, can afford a fine-tuning pass, and
either want structured shrinkage or are on hardware with 2:4 support.

**Don't** reach for it when you want a quick memory win — [quantization](quantization-gptq-awq.ipynb)
gives you 2–4× for a fraction of the effort and with no shape changes.

## 2. Mental Model

**Pruning a tree.** You can take out twigs from all over the canopy, or you can cut off
a whole branch.

Taking out scattered twigs (**unstructured** pruning) barely changes the tree's shape.
You can remove a lot of them before it looks different — which is exactly why
unstructured pruning preserves accuracy so well at high sparsity. But the tree still
occupies the same volume: nothing about the *bounding box* changed, and on a GPU the
bounding box is what you pay for.

Cutting a branch (**structured** pruning) visibly changes the shape. The tree is
genuinely smaller and it is much easier to see when you have cut too much — structured
pruning hurts accuracy sooner. But it is the only cut that makes the thing smaller in a
way a dense matmul can see.

**2:4 semi-structured** is the compromise the hardware was built for: scattered twigs,
but with a rule about how they may be scattered (exactly two of every four must go), so
the silicon can skip them.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Sparsity ratio** | Fraction of weights set to zero. Reported per-layer or globally — they are not the same number. |
| **Unstructured** | Any weight may go. Best accuracy at a given ratio; no dense-hardware speedup. |
| **Structured** | Whole rows / output channels / attention heads / layers go. Real speedup; accuracy falls faster. |
| **Semi-structured (N:M)** | N zeros in every contiguous group of M. 2:4 is supported by NVIDIA sparse tensor cores. |
| **Magnitude criterion** | Rank weights by \|w\|, prune the smallest. The default, and a very strong baseline. |
| **Global vs layerwise** | One threshold across the whole net, versus a separate ratio per layer. Global spends the budget where it is cheapest. |
| **One-shot vs iterative** | Prune once, or prune-and-recover in several rounds. Iterative reaches higher sparsity. |
| **Fine-tune after prune** | The recovery pass. Skipping it is the most common reason pruning "doesn't work". |
| **Lottery Ticket Hypothesis** | A dense net contains a sparse subnetwork that, *rewound to its initialisation*, trains to comparable accuracy alone. |
| **Movement pruning** | Importance = how far a weight moved *away from zero* during fine-tuning, not its magnitude. Better under transfer learning. |
| **Wanda / SparseGPT** | Modern one-shot LLM pruning that weights the criterion by input activation scale, so no retraining is needed. |

## 4. Setup

Pure NumPy — the criteria and the mask arithmetic are the whole idea, and they are
short enough to write out. The PyTorch API shape appears in Example 4, gated so this
notebook runs without it.

In [1]:
# %pip install numpy
# Optional, for the gated Example 4:
# %pip install torch

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

try:
    import torch
    HAVE_TORCH = True
    print("torch", torch.__version__)
except ImportError:
    HAVE_TORCH = False
    print("torch not installed - Example 4 prints its API shape instead of running")

numpy 2.5.1
torch not installed - Example 4 prints its API shape instead of running


## 5. Worked Examples

### Example 1 — global beats layerwise, and here is by how much

Two layers with very different weight scales. Layerwise pruning removes the same
*fraction* from each; global pruning pools all the weights and removes the smallest
overall. The comparison is the relative error each introduces.

In [2]:
# Two layers whose weights live on different scales -- extremely common in practice
# (early layers small, later layers large, or vice versa after normalisation).
W_small = rng.standard_normal((128, 128)) * 0.01
W_large = rng.standard_normal((128, 128)) * 1.00
layers = {"layer_a (scale 0.01)": W_small, "layer_b (scale 1.00)": W_large}

def prune_layerwise(layers, ratio):
    out = {}
    for name, W in layers.items():
        k = int(round(ratio * W.size))
        thresh = np.partition(np.abs(W).ravel(), k)[k] if k else 0.0
        out[name] = W * (np.abs(W) >= thresh)
    return out

def prune_global(layers, ratio):
    allw = np.concatenate([np.abs(W).ravel() for W in layers.values()])
    k = int(round(ratio * allw.size))
    thresh = np.partition(allw, k)[k] if k else 0.0
    return {name: W * (np.abs(W) >= thresh) for name, W in layers.items()}

def rel_err(orig, pruned):
    return float(np.linalg.norm(orig - pruned) / np.linalg.norm(orig))

def sparsity(W):
    return float((W == 0).mean())

RATIO = 0.8
lw, gl = prune_layerwise(layers, RATIO), prune_global(layers, RATIO)

print(f"target sparsity {RATIO:.0%}\n")
print(f"{'':22} {'layerwise':>22}  {'global':>22}")
print(f"{'':22} {'sparsity   rel.err':>22}  {'sparsity   rel.err':>22}")
for name, W in layers.items():
    print(f"{name:22} {sparsity(lw[name]):8.0%} {rel_err(W, lw[name]):9.4f}  "
          f"{sparsity(gl[name]):8.0%} {rel_err(W, gl[name]):9.4f}")

tot = lambda d: rel_err(np.concatenate([W.ravel() for W in layers.values()]),
                        np.concatenate([d[n].ravel() for n in layers]))
print(f"\n{'WHOLE MODEL':22} {tot(lw):18.4f}  {tot(gl):22.4f}")
print("\nGlobal pruning notices that layer_a's weights are all tiny and sacrifices")
print("almost all of them, spending its budget where the error is cheapest.")

target sparsity 80%

                                    layerwise                  global
                           sparsity   rel.err      sparsity   rel.err
layer_a (scale 0.01)        80%    0.5932      100%    1.0000
layer_b (scale 1.00)        80%    0.5906       60%    0.3582

WHOLE MODEL                        0.5906                  0.3583

Global pruning notices that layer_a's weights are all tiny and sacrifices
almost all of them, spending its budget where the error is cheapest.


That result is also the warning: global pruning happily removes *an entire layer* if
that layer's weights happen to be small, which is catastrophic rather than cheap. Most
real recipes use global ranking with a per-layer floor.

### Example 2 — 2:4 semi-structured, and what the constraint costs

The 2:4 pattern keeps the two largest magnitudes in every group of four. It is fixed at
50% sparsity by construction. The question is how much accuracy you give up versus
unstructured pruning at the same 50%.

In [3]:
def prune_2of4(W):
    '''Keep the 2 largest-|w| entries in every contiguous group of 4 along the last axis.'''
    flat = W.reshape(-1, 4)
    keep = np.argsort(-np.abs(flat), axis=1)[:, :2]          # indices of the top 2
    mask = np.zeros_like(flat, dtype=bool)
    np.put_along_axis(mask, keep, True, axis=1)
    return (flat * mask).reshape(W.shape)

def prune_unstructured(W, ratio):
    k = int(round(ratio * W.size))
    thresh = np.partition(np.abs(W).ravel(), k)[k] if k else 0.0
    return W * (np.abs(W) >= thresh)

W = rng.standard_normal((512, 512))
u50 = prune_unstructured(W, 0.50)
s24 = prune_2of4(W)

print(f"unstructured 50%   sparsity {sparsity(u50):.1%}   rel.err {rel_err(W, u50):.4f}")
print(f"2:4              sparsity {sparsity(s24):.1%}   rel.err {rel_err(W, s24):.4f}")
print(f"\n2:4 costs {100*(rel_err(W, s24)/rel_err(W, u50) - 1):.0f}% more error than "
      f"unconstrained pruning at the same 50% ...")
print("... and unlike unconstrained pruning, it actually runs ~2x faster on Ampere+")
print("sparse tensor cores. That trade is why 2:4 is the default for LLM pruning.")

# Why the gap is small: in a group of 4 iid draws, the smallest two are usually
# genuinely small. The constraint only bites in groups that are all-large.
groups = np.abs(W).reshape(-1, 4)
all_large = (groups > np.median(np.abs(W))).all(axis=1).mean()
print(f"\nonly {all_large:.1%} of groups have all four weights above the median,")
print("so the 2:4 rule rarely forces out a weight that unstructured pruning would keep.")

unstructured 50%   sparsity 50.0%   rel.err 0.2672
2:4              sparsity 50.0%   rel.err 0.3638

2:4 costs 36% more error than unconstrained pruning at the same 50% ...
... and unlike unconstrained pruning, it actually runs ~2x faster on Ampere+
sparse tensor cores. That trade is why 2:4 is the default for LLM pruning.

only 6.2% of groups have all four weights above the median,
so the 2:4 rule rarely forces out a weight that unstructured pruning would keep.


### Example 3 — the one that matters: does the matmul get faster?

Unstructured pruning changes the *values* in the matrix; structured pruning changes its
**shape**. Only the second one is visible to a dense matmul. This measures wall-clock.

In [4]:
import time

def bench(A, B, repeats=20):
    A @ B                                        # warm up
    t0 = time.perf_counter()
    for _ in range(repeats):
        A @ B
    return (time.perf_counter() - t0) / repeats * 1e3   # ms

N = 1024
X = rng.standard_normal((N, N))
W_dense = rng.standard_normal((N, N))

W_unstruct = prune_unstructured(W_dense, 0.90)          # 90% zeros, same shape
keep_rows = np.argsort(-np.linalg.norm(W_dense, axis=1))[: N // 10]   # top 10% of rows
W_struct = W_dense[np.sort(keep_rows)]                  # 90% smaller, real shape change

t_dense = bench(X, W_dense)
t_unstruct = bench(X, W_unstruct)
t_struct = bench(X[:, np.sort(keep_rows)], W_struct)

print(f"{'variant':24} {'shape':>14} {'zeros':>8} {'ms':>8} {'speedup':>9}")
print(f"{'dense':24} {str(W_dense.shape):>14} {sparsity(W_dense):8.0%} "
      f"{t_dense:8.2f} {1.0:8.2f}x")
print(f"{'unstructured 90%':24} {str(W_unstruct.shape):>14} {sparsity(W_unstruct):8.0%} "
      f"{t_unstruct:8.2f} {t_dense/t_unstruct:8.2f}x")
print(f"{'structured (rows) 90%':24} {str(W_struct.shape):>14} {sparsity(W_struct):8.0%} "
      f"{t_struct:8.2f} {t_dense/t_struct:8.2f}x")

print("\nThe unstructured matrix is 90% zeros and takes essentially the same time:")
print("a dense kernel has no idea the zeros are there. The structured one is a")
print("genuinely smaller matrix, so it is genuinely faster.")
print("\nIf you only ever report sparsity %, these two look identical. They are not.")

variant                           shape    zeros       ms   speedup
dense                      (1024, 1024)       0%     2.91     1.00x
unstructured 90%           (1024, 1024)      90%     2.91     1.00x
structured (rows) 90%       (102, 1024)       0%     0.40     7.23x

The unstructured matrix is 90% zeros and takes essentially the same time:
a dense kernel has no idea the zeros are there. The structured one is a
genuinely smaller matrix, so it is genuinely faster.

If you only ever report sparsity %, these two look identical. They are not.


### Example 4 — activation-aware criteria, and the PyTorch API

Pure magnitude ignores a crucial fact: a small weight multiplying a large activation
matters more than a large weight multiplying a near-zero one. Wanda's criterion is
`|w| · ‖x‖`, and it is a one-line change that needs no retraining.

In [5]:
# A calibration batch with wildly different input scales per feature -- typical of
# transformer activations, where a few channels carry outsized magnitude.
X_calib = rng.standard_normal((64, 256)) * np.logspace(-2, 1, 256)
W_layer = rng.standard_normal((256, 256)) * 0.1

act_norm = np.linalg.norm(X_calib, axis=0)                # per-input-feature scale
magnitude_score = np.abs(W_layer)
wanda_score = np.abs(W_layer) * act_norm[:, None]         # |w| * ||x||

def prune_by_score(W, score, ratio):
    k = int(round(ratio * W.size))
    thresh = np.partition(score.ravel(), k)[k]
    return W * (score >= thresh)

Y_true = X_calib @ W_layer
for name, score in [("magnitude", magnitude_score), ("Wanda |w|*||x||", wanda_score)]:
    Wp = prune_by_score(W_layer, score, 0.70)
    err = np.linalg.norm(Y_true - X_calib @ Wp) / np.linalg.norm(Y_true)
    print(f"{name:18} 70% pruned -> output rel.err {err:.4f}")

print("\nSame sparsity, same weights. Ranking by the weight's actual *effect* on the")
print("output beats ranking by its size, because what you care about is the output.")

TORCH_PRUNING = '''
import torch.nn.utils.prune as prune

# unstructured, one layer
prune.l1_unstructured(model.fc, name="weight", amount=0.8)

# global across many layers -- the Example 1 result
prune.global_unstructured(
    [(m, "weight") for m in model.modules() if isinstance(m, torch.nn.Linear)],
    pruning_method=prune.L1Unstructured, amount=0.8,
)

# structured: drop whole output channels (dim=0) by L2 norm
prune.ln_structured(model.conv, name="weight", amount=0.5, n=2, dim=0)

prune.remove(model.fc, "weight")   # bake the mask in; without this it stays a hook

# 2:4 for Ampere+ sparse tensor cores
from torch.sparse import to_sparse_semi_structured
dense = prune_2of4_(model.fc.weight.data)
model.fc.weight = torch.nn.Parameter(to_sparse_semi_structured(dense))
'''
print("\n--- PyTorch API shape ---")
print(TORCH_PRUNING if not HAVE_TORCH else TORCH_PRUNING)

magnitude          70% pruned -> output rel.err 0.4614
Wanda |w|*||x||    70% pruned -> output rel.err 0.0700

Same sparsity, same weights. Ranking by the weight's actual *effect* on the
output beats ranking by its size, because what you care about is the output.

--- PyTorch API shape ---

import torch.nn.utils.prune as prune

# unstructured, one layer
prune.l1_unstructured(model.fc, name="weight", amount=0.8)

# global across many layers -- the Example 1 result
prune.global_unstructured(
    [(m, "weight") for m in model.modules() if isinstance(m, torch.nn.Linear)],
    pruning_method=prune.L1Unstructured, amount=0.8,
)

# structured: drop whole output channels (dim=0) by L2 norm
prune.ln_structured(model.conv, name="weight", amount=0.5, n=2, dim=0)

prune.remove(model.fc, "weight")   # bake the mask in; without this it stays a hook

# 2:4 for Ampere+ sparse tensor cores
from torch.sparse import to_sparse_semi_structured
dense = prune_2of4_(model.fc.weight.data)
model.fc.weight = tor

## 6. Gotchas & Pitfalls

- **Reporting sparsity instead of latency.** Example 3 is the whole warning. "90%
  sparse" with no shape change and no sparse kernel is a 1.0× speedup. Always measure
  the thing you actually want.
- **Masks cost memory.** A boolean mask alongside the dense weights makes the model
  *bigger* until you actually compact the tensors. `prune.remove()` in PyTorch bakes
  the mask into the weights; it does not shrink the tensor.
- **Skipping the fine-tune.** One-shot pruning past ~50% without recovery usually falls
  off a cliff. The exceptions are the activation-aware LLM methods (Wanda, SparseGPT),
  which are explicitly designed to be retraining-free.
- **Pruning the wrong parameters.** Leave LayerNorm/BatchNorm scales, biases and
  embeddings alone by default. They are a trivial share of the parameters and pruning
  them does disproportionate damage.
- **Global thresholds that delete a layer.** As in Example 1 — one badly-scaled layer
  can absorb the whole budget. Use a per-layer floor.
- **Assuming a pruned model fine-tunes like a dense one.** The mask must be re-applied
  after every optimiser step, or the pruned weights simply come back from zero.
- **Comparing against the wrong baseline.** A model pruned to 50% should be compared
  against a *dense model of the same final size*, not against the original. Very often
  training the smaller dense model outright wins — that is the real lesson of much of
  the lottery-ticket literature.
- **2:4 on the wrong hardware.** The 2× is a sparse-tensor-core feature. On anything
  before Ampere, 2:4 is pure accuracy loss for nothing.

## 7. When to Use vs Alternatives

| Goal | Reach for |
|---|---|
| Cheapest real memory + latency win | [Quantization](quantization-gptq-awq.ipynb) — always try this first |
| A genuinely smaller/faster model shape | **Structured pruning**, or [distillation](knowledge-distillation.ipynb) |
| ~2× on Ampere+ with modest accuracy loss | **2:4 semi-structured pruning** |
| Maximum sparsity, and you own training | [Sparsity induction](sparsity-induction.ipynb) — beats post-hoc pruning at the same ratio |
| Shrinking specific big matrices | [Low-rank factorization](low-rank-factorization.ipynb) |

**The honest ordering.** Quantize first. If you still need more, pick structured
pruning or distillation depending on whether you want the same architecture smaller
(pruning) or a different architecture entirely (distillation). Reach for unstructured
pruning only when you have a sparse kernel that will actually exploit it, or when you
are pruning for *memory footprint* on a format that stores sparse tensors compactly.

Pruning composes cleanly with quantization — prune, fine-tune, then quantize — and with
distillation, where the pruned model is often a better student initialisation than a
randomly-initialised small model.

## 8. Resources

- [Learning both Weights and Connections for Efficient Neural Networks](https://arxiv.org/abs/1506.02626) — Han et al., 2015. The paper that made magnitude pruning standard.
- [The Lottery Ticket Hypothesis](https://arxiv.org/abs/1803.03635) — Frankle & Carbin. Why the sparse subnetwork matters and why rewinding to initialisation is the surprising part.
- [A Simple and Effective Pruning Approach for Large Language Models](https://arxiv.org/abs/2306.11695) — Wanda; the `|w|·‖x‖` criterion from Example 4.
- [SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot](https://arxiv.org/abs/2301.00774) — one-shot 50% on 100B+ models with no retraining.
- [Movement Pruning: Adaptive Sparsity by Fine-Tuning](https://arxiv.org/abs/2005.07683) — why magnitude is the wrong criterion during transfer learning.
- [PyTorch pruning tutorial](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial.html) — the `torch.nn.utils.prune` API in full.
- [Accelerating Inference with Sparsity Using Ampere and TensorRT](https://developer.nvidia.com/blog/accelerating-inference-with-sparsity-using-ampere-and-tensorrt/) — what 2:4 actually buys on real hardware.